#  Segmentation (U-Net)


In [3]:
import os
from pathlib import Path
import random

import numpy as np
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DEVICE

/Users/jim/anaconda3/envs/pytorch/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


device(type='cpu')

In [ ]:
# Config\n
DATA_ROOT = Path("/Users/jim/Downloads/competition_data")
IMAGE_DIR = DATA_ROOT / "train/images"
MASK_DIR = DATA_ROOT / "train/masks"

IMG_SIZE = 101
BATCH_SIZE = 8
LR = 1e-3
WEIGHT_DECAY = 1e-5
EPOCHS = 20
VAL_RATIO = 0.1

assert IMAGE_DIR.exists(), f"找不到影像資料夾: {IMAGE_DIR}"
assert MASK_DIR.exists(), f"找不到遮罩資料夾: {MASK_DIR}"

In [24]:
imageSample = Image.open('/Users/jim/Downloads/competition_data/train/images/000e218f21.png')
imageSample.show()

In [ ]:
class CarvanaDataset(Dataset):
    def __init__(self, image_dir: Path, mask_dir: Path, img_size: int = 256):
        self.image_dir = image_dir
        self.mask_dir = mask_dir
        self.images = sorted(list(image_dir.glob("*.png")))

        self.pairs = []
        for img_path in self.images:
            mask_path = mask_dir / f"{img_path.stem}.png"
            if mask_path.exists():
                self.pairs.append((img_path, mask_path))

        if len(self.pairs) == 0:
            raise RuntimeError("沒有找到任何可配對的影像/遮罩，請檢查檔名與資料夾")

        self.image_tf = transforms.Compose([
            transforms.Resize((img_size, img_size)),
            transforms.ToTensor(),
        ])
        self.mask_tf = transforms.Compose([
            transforms.Resize((img_size, img_size), interpolation=transforms.InterpolationMode.NEAREST),
            transforms.ToTensor(),
        ])

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        img_path, mask_path = self.pairs[idx]
        image = Image.open(img_path).convert("RGB")
        mask = Image.open(mask_path).convert("L")

        image = self.image_tf(image)
        mask = self.mask_tf(mask)
        mask = (mask > 0.5).float()
        return image, mask

dataset = CarvanaDataset(IMAGE_DIR, MASK_DIR, img_size=IMG_SIZE)
val_len = int(len(dataset) * VAL_RATIO)
train_len = len(dataset) - val_len


train_set, val_set = random_split(dataset, [train_len, val_len], generator=torch.Generator().manual_seed(SEED))

train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True,  pin_memory=True)
val_loader = DataLoader(val_set, batch_size=BATCH_SIZE, shuffle=False, pin_memory=True)

len(dataset), len(train_set), len(val_set)

(4000, 3600, 400)

In [31]:
class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)

class UNet(nn.Module):
    def __init__(self, in_ch=3, out_ch=1, features=(16, 32, 64)):
        super().__init__()
        self.downs = nn.ModuleList()
        self.ups = nn.ModuleList()
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

        ch = in_ch
        for f in features:
            self.downs.append(DoubleConv(ch, f))
            ch = f

        self.bottleneck = DoubleConv(features[-1], features[-1] * 2)

        rev = features[::-1]
        up_in = features[-1] * 2
        for f in rev:
            self.ups.append(nn.ConvTranspose2d(up_in, f, kernel_size=2, stride=2))
            self.ups.append(DoubleConv(f * 2, f))
            up_in = f

        self.final_conv = nn.Conv2d(features[0], out_ch, kernel_size=1)
    def forward(self, x):
        skips = []
        for down in self.downs:
            x = down(x)
            skips.append(x)
            x = self.pool(x)

        x = self.bottleneck(x)
        skips = skips[::-1]

        for idx in range(0, len(self.ups), 2):
            x = self.ups[idx](x)
            skip = skips[idx // 2]
            if x.shape[-2:] != skip.shape[-2:]:
                x = F.interpolate(x, size=skip.shape[-2:], mode="bilinear", align_corners=False)
            x = torch.cat((skip, x), dim=1)
            x = self.ups[idx + 1](x)

        return self.final_conv(x)

def dice_coefficient(logits, targets, eps=1e-6):
    probs = torch.sigmoid(logits)
    preds = (probs > 0.5).float()
    intersection = (preds * targets).sum(dim=(1, 2, 3))
    union = preds.sum(dim=(1, 2, 3)) + targets.sum(dim=(1, 2, 3))
    dice = (2 * intersection + eps) / (union + eps)
    return dice.mean().item()

In [32]:
model = UNet().to(DEVICE)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

best_val_dice = 0.0
save_path = Path("unet_carvana_best.pth")

for epoch in range(1, EPOCHS + 1):
    model.train()
    train_loss = 0.0

    for images, masks in tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS} [Train]"):
        images = images.to(DEVICE, non_blocking=True)
        masks = masks.to(DEVICE, non_blocking=True)

        optimizer.zero_grad()
        logits = model(images)
        loss = criterion(logits, masks)
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * images.size(0)

    train_loss /= len(train_loader.dataset)

    model.eval()
    val_loss = 0.0
    val_dice_sum = 0.0

    with torch.no_grad():
        for images, masks in tqdm(val_loader, desc=f"Epoch {epoch}/{EPOCHS} [Val]"):
            images = images.to(DEVICE, non_blocking=True)
            masks = masks.to(DEVICE, non_blocking=True)

            logits = model(images)
            loss = criterion(logits, masks)

            val_loss += loss.item() * images.size(0)
            val_dice_sum += dice_coefficient(logits, masks) * images.size(0)

    val_loss /= len(val_loader.dataset)
    val_dice = val_dice_sum / len(val_loader.dataset)

    print(f"Epoch {epoch:02d} | train_loss: {train_loss:.4f} | val_loss: {val_loss:.4f} | val_dice: {val_dice:.4f}")

    if val_dice > best_val_dice:
        best_val_dice = val_dice
        torch.save(model.state_dict(), save_path)
        print(f"  Saved best model to {save_path} (dice={best_val_dice:.4f})")

print(f"Training done. Best val dice: {best_val_dice:.4f}")

Epoch 1/20 [Val]: 100%|██████████| 50/50 [01:03<00:00,  1.26s/it]


Epoch 01 | train_loss: 0.4445 | val_loss: 0.4359 | val_dice: 0.5968
  Saved best model to unet_carvana_best.pth (dice=0.5968)


Epoch 2/20 [Val]: 100%|██████████| 50/50 [00:59<00:00,  1.18s/it]


Epoch 02 | train_loss: 0.3509 | val_loss: 0.3136 | val_dice: 0.6431
  Saved best model to unet_carvana_best.pth (dice=0.6431)


Epoch 3/20 [Val]: 100%|██████████| 50/50 [01:13<00:00,  1.46s/it]


...

Epoch 04 | train_loss: 0.2996 | val_loss: 0.2421 | val_dice: 0.7013
  Saved best model to unet_carvana_best.pth (dice=0.7013)


Epoch 5/20 [Val]: 100%|██████████| 50/50 [01:15<00:00,  1.51s/it]


Epoch 05 | train_loss: 0.2825 | val_loss: 0.2219 | val_dice: 0.6821


Epoch 6/20 [Train]:  20%|██        | 91/450 [04:27<17:36,  2.94s/it]


KeyboardInterrupt: 

In [ ]:
# test data loader
from pathlib import Path

TEST_DIR = Path('/Users/jim/Downloads/competition_data/test/images')

class TestDataset(Dataset):
    def __init__(self, test_dir: Path, img_size: int = IMG_SIZE):
        self.images = list(test_dir.glob('*.png'))
        self.image_tf = transforms.Compose([
            transforms.Resize((img_size, img_size)),
            transforms.ToTensor(),
        ])

    def __len__(self):
        return len(self.images)

    def __getitem__(self, index):
        img_path = self.images[index]
        image = Image.open(img_path).convert('RGB')
        image = self.image_tf(image)
        image_id = img_path.stem
        return image, image_id


test_dataset = TestDataset(TEST_DIR)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)
len(test_dataset)



18000

In [35]:
model.load_state_dict(torch.load(save_path, map_location=DEVICE))
model.eval()

all_masks = {}

with torch.no_grad():
    for images, ids in tqdm(test_loader, desc='Inference'):
        images = images.to(DEVICE, non_blocking=True)
        logits = model(images)
        probs = torch.sigmoid(logits)
        preds = (probs > 0.5).float().cpu().numpy()  # [B, 1, H, W]

        for i, image_id in enumerate(ids):
            all_masks[image_id] = preds[i, 0]

len(all_masks)



/var/folders/jq/nkg3fdnj72dbspmxh6k5gbj40000gn/T/ipykernel_22700/3214768578.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(save_path, m

18000

In [ ]:
def mask_to_rle(mask):
    mask = np.asarray(mask).astype(np.uint8)
    pixels = mask.flatten(order='F')
    pixels = np.concatenate([[0], pixels, [0]])#
    runs = np.where(pixels[1:] != pixels[:-1])[0] + 1
    runs[1::2] -= runs[::2]
    return ' '.join(str(x) for x in runs)


In [39]:
import pandas as pd

submission = pd.DataFrame({
    'id': list(all_masks.keys()),
    'rle_mask': [mask_to_rle(all_masks[k]) for k in all_masks.keys()]
})

submission.to_csv('submission.csv', index=False)
submission.head()



,id,rle_mask
0,0005bb9630,
1,000a68e46c,1 49 102 49 203 49 304 49 405 48 498 1 506 48 ...
2,000c8dfb2a,1 100 102 2021 2124 100 2225 100 2326 100 2427...
3,000d0a5f6c,29 37 128 40 235 33 338 30 440 28 542 19 562 6...
4,001ef8fc87,1 7170 7172 85 7267 2 7273 81 7374 78 7475 1 7...
